In [ ]:
# Detect if running in Google Colab
try:
    from google.colab import drive
    colab_nb = True
except ImportError:
    colab_nb = False

In [ ]:
# Mount Google Drive (only runs in Colab; no effect if running locally)
if colab_nb:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

In [ ]:
# Install aad package from Google Drive (only runs in Colab; no effect if running locally)
if colab_nb:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "/content/drive/MyDrive/aad"])

# Lane Detector

In this exercise we will implement the polynomial fitting and then combine all functionality into one `LaneDetector` class

## Precompute the grid

In the book you have seen how the tensor `xyp` was computed. Its first two columns have the `x` and `y` values respectively, while the last column has the probability values. The `x` and `y` values will always be the same. Hence they only need to be computed once.
This is what you should implement first. It is marked as "TODO step 3" in `aad/exercises/lane_detection/camera_geometry.py`. Note that there is one additional modification to what you have seen in the book: The `cut_v` parameter. In the book the `(x,y,p)` triples were computed from all possible `u, v, p[v,u]`. Here you should restrict yourself to all `v` with `v>cut_v`. The idea is that pixels with low `v` values are too far away or even above the horizon, and hence should not be considered for fitting later. The other modification is of course that you do not need to compute an `xyp` tensor, since you have no probabilities given. You only precompute the first two columns of the `xyp` tensor.

Once you implemented "TODO step 3", check whether your implementation is correct using the unit test:

In [5]:
# execute this cell to run unit tests on your implementation of step 3
%cd ../../../
!uv run python -m aad.tests.lane_detection.camera_geometry_unit_test 3
%cd -

/home/mtheers/repos/Algorithms-for-Automated-Driving
-------------------------
Running tests for step  3
-------------------------
ERROR:root:An exception was thrown in your CameraGeometry class! I will show you the traceback:
Traceback (most recent call last):
  File "/home/mtheers/repos/Algorithms-for-Automated-Driving/code/tests/lane_detection/camera_geometry_unit_test.py", line 127, in <module>
    ex_cg = ex_CameraGeometry()
  File "/home/mtheers/repos/Algorithms-for-Automated-Driving/code/exercises/lane_detection/camera_geometry.py", line 49, in __init__
    self.intrinsic_matrix = get_intrinsic_matrix(field_of_view_deg, image_width, image_height)
                            ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mtheers/repos/Algorithms-for-Automated-Driving/code/exercises/lane_detection/camera_geometry.py", line 12, in get_intrinsic_matrix
    raise NotImplementedError
NotImplementedError
/home/mtheers/repos/Algorithms-for-Automated-Dri

## Implement the LaneDetector class

Your final step is to implement the LaneDetector class. 

1. Read the rest of this notebook. You will find places where it says "TODO" and you are asked to change something. Do not do this yet! For now, you should just see the sample solution at work.
2. Go to `aad/exercises/lane_detection/lane_detector.py` and implement the "TODO" items. 
3. Now it is time to test **your** lane detector. Go through all the cells below and execute them. Some cells will have a "TODO". Please resolve them, so that your lane detector is being run.

Does your `LaneDetector` class work to your satisfaction? If not, debug and improve it!

In [6]:
import numpy as np 
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
from pathlib import Path
import cv2

AttributeError: module 'IPython.display' has no attribute 'set_matplotlib_formats'

In [ ]:

# TODO: In the next two lines, change "solutions" to "exercises". Now your code will be executed here!
from aad.solutions.lane_detection.lane_detector import LaneDetector
from aad.solutions.lane_detection.camera_geometry import CameraGeometry
cg = CameraGeometry()

In [7]:
image_fn = str(Path("../../../data/Town04_Clear_Noon_09_09_2020_14_57_22_frame_625_validation_set.png").absolute())
image_arr = cv2.imread(image_fn)
image_arr = cv2.cvtColor(image_arr, cv2.COLOR_BGR2RGB)
plt.imshow(image_arr);

NameError: name 'Path' is not defined

### Get lane boundaries from LaneDetector

In [8]:
# TODO: Change the next line(s), to create an instance of *your* LaneDetector
model_path = Path("../../solutions/lane_detection/fastai_model.pth")
ld = LaneDetector(model_path=model_path)
poly_left, poly_right = ld(image_fn)

NameError: name 'Path' is not defined

In [9]:
# It should also be possible to pass the image as an array into the lane detector
# The following assertions should not raise an AssertionError
poly_left_2, poly_right_2 = ld(image_arr)
np.testing.assert_allclose(poly_left, poly_left_2, rtol=1e-5)
np.testing.assert_allclose(poly_right, poly_right_2, rtol=1e-5)
# we are using `assert_allclose` to compare floating point numbers here.

NameError: name 'ld' is not defined

In [10]:
# Let's see how fast the lane detector works:
%timeit poly_left, poly_right = ld(image_arr)

NameError: name 'ld' is not defined

### Get ground truth for lane boundaries

In [11]:
boundary_fn = image_fn.replace(".png", "_boundary.txt")
boundary_gt = np.loadtxt(boundary_fn)

trafo_fn = image_fn.replace(".png", "_trafo.txt")
trafo_world_to_cam = np.loadtxt(trafo_fn)

NameError: name 'image_fn' is not defined

In [ ]:
# Map reconstructed left boundary into world reference frame
def map_between_frames(points, trafo_matrix):
    x,y,z = points[:,0], points[:,1], points[:,2]
    homvec = np.stack((x,y,z,np.ones_like(x)))
    return (trafo_matrix @ homvec).T

trafo_world_to_road = cg.trafo_cam_to_road @ trafo_world_to_cam

In [ ]:
left_boundary_3d_gt_world = boundary_gt[:,0:3]

left_boundary_gt_road = map_between_frames(boundary_gt[:,0:3], trafo_world_to_road)
right_boundary_gt_road = map_between_frames(boundary_gt[:,3:], trafo_world_to_road)

### Plot LaneDetector output and ground truth

In [ ]:
# ground truth
plt.plot(left_boundary_gt_road[:,2], -left_boundary_gt_road[:,0], label="ground truth left")
plt.plot(right_boundary_gt_road[:,2], -right_boundary_gt_road[:,0], label="ground truth right")
# LaneDetector
x = np.arange(0,60,1)
yl = poly_left(x)
yr = poly_right(x)
plt.plot(x,yl, ls = "--", label="LaneDector left")
plt.plot(x,yr, ls = "--", label="LaneDector right")
plt.legend()
# TODO: You can also inspect the plot while commenting out the next line
#plt.axis("equal");

In the plot above, the LaneDetector should yield something close to the ground truth (less than 1m error along the y axis). 